# README

- **Author**: 楊喬茵（AI協助：Claude Code）
- **Created At**: 2026-11-04
- **Last Modified At**: 2026-11-04

---

## What does this file do?

- 為臺北市與高雄市2026直轄市議員候選人建立「人工核對工作單」：觀察單位為「候選人 × 缺值欄位」，只列 `candidates_g1_g6_current.csv` 中 G1、G3、G5、G6 為空值的格子（G2、G4 兩市皆無缺值）。
- 不填補、不推定任何值；空值仍代表「未知」，不是0。工作單提供核對指引、既有線索（同名同選區歷史生日）與空白回填欄，供組員人工查證後回填。
- 優先序：G1／G3／G5 為 `high`，G6 為 `low`（G6只有找到配偶或三親等內親屬任公職的證據才可填1，查無不可填0）。
- 依學生2026-11-04確認：新聞只寫「現年XX歲」時可用，但須在 `age_derived_flag` 標示「由報導年齡推算±1」。

---

## What does this file take?

- **Source Data Sets** (one entry per actual file):
  1. `data/processed/candidates_g1_g6_current.csv`
     - Input key: `g1_g6_current`
     - Description: 2026登記候選人G1–G6主表，1,583列×36欄，唯一鍵 `candidate_key`。
     - Documentation: `data/processed/g1_g6_20261103_documentation.ipynb`
     - SHA-256: `790407203ac8d7d4de3bc605fe30fe883007902978f89fa416e7e5593456de02`
  2. `data/processed/candidate_g1_g6_identity_pending_20261103.csv`
     - Input key: `identity_pending`
     - Description: 只有同名同縣市選區支持、未升級為同一人的歷史紀錄（含歷史生日），121列。
     - Documentation: `data/processed/g1_g6_20261103_documentation.ipynb`
     - SHA-256: `56ecdfc28c05ab7fc3e5a343f82ad3c393be9d8306a119fea1a2bc129580dc36`

---

## What does this file output?

- `data/processed/manual_check_worklist_taipei_kaohsiung_20261104.csv`
  - Output key: `manual_worklist`
  - Description: 臺北市、高雄市議員候選人缺值格子的人工核對工作單，含空白回填欄。
  - Role: collection worklist（不是分析輸入）
  - Documentation: 本筆記本（欄位說明見最後一格）
  - Expected schema and row-count rules: 每列一個（candidate_key, field）且唯一；列數等於兩市議員G1、G3、G5、G6空值總數；回填欄全空。
  - Verification status / SHA-256: 見執行輸出
  - Downstream finding input: 無；人工回填後需另以筆記本整合並重新驗證，才可更新主表。

All paths are relative to the project root, not the operating-system root.


In [1]:
from pathlib import Path
import hashlib
import pandas as pd


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "AGENTS.md").is_file() and (candidate / "templates").is_dir():
            return candidate
    raise FileNotFoundError("Open this notebook within the student project containing AGENTS.md and templates/.")


PROJECT_ROOT = find_project_root()


def project_file(relative_path, area, must_exist=True):
    # Explicit project-relative paths only; never select the first matching file.
    rel = Path(relative_path)
    if rel.is_absolute() or ".." in rel.parts or "<" in str(rel):
        raise ValueError(f"Replace the placeholder with an exact project-relative path: {relative_path}")
    path = (PROJECT_ROOT / rel).resolve()
    allowed = (PROJECT_ROOT / "data" / area).resolve()
    if allowed == PROJECT_ROOT or not allowed.is_relative_to(PROJECT_ROOT):
        raise ValueError(f"data/{area}/ resolves outside the project root.")
    if not path.is_relative_to(allowed):
        raise ValueError(f"Expected a file inside data/{area}/: {relative_path}")
    if must_exist and not path.is_file():
        raise FileNotFoundError(path)
    return path


def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def verify_input_sha256(path, expected):
    # Bind this run to the exact file inspected when the notebook was prepared.
    if (not isinstance(expected, str) or len(expected) != 64
            or any(char not in "0123456789abcdefABCDEF" for char in expected)):
        raise ValueError("Fill the expected SHA-256 from the inspected input before loading data.")
    actual = sha256_file(path)
    if actual != expected.lower():
        raise ValueError(f"Input changed: identify the new version before loading {path.name}.")
    return actual


In [2]:
INPUTS = {"g1_g6_current": "data/processed/candidates_g1_g6_current.csv",
          "identity_pending": "data/processed/candidate_g1_g6_identity_pending_20261103.csv"}
EXPECTED_INPUT_SHA256 = {"g1_g6_current": "790407203ac8d7d4de3bc605fe30fe883007902978f89fa416e7e5593456de02",
                         "identity_pending": "56ecdfc28c05ab7fc3e5a343f82ad3c393be9d8306a119fea1a2bc129580dc36"}
OUTPUTS = {"manual_worklist": "data/processed/manual_check_worklist_taipei_kaohsiung_20261104.csv"}
INPUT_OPTIONS = {"g1_g6_current": {"dtype": str, "encoding": "utf-8-sig"},
                 "identity_pending": {"dtype": str, "encoding": "utf-8-sig"}}
OUTPUT_READ_OPTIONS = {"manual_worklist": {"dtype": str, "encoding": "utf-8-sig", "keep_default_na": False}}

input_paths = {}
for key, relative_path in INPUTS.items():
    area = Path(relative_path).parts[1] if len(Path(relative_path).parts) > 1 else ""
    if area not in {"raw", "processed"}:
        raise ValueError(f"Expected a raw or upstream processed input: {relative_path}")
    input_paths[key] = project_file(relative_path, area)
output_paths = {key: project_file(path, "processed", must_exist=False) for key, path in OUTPUTS.items()}
if len(set(output_paths.values())) != len(output_paths):
    raise ValueError("Each output needs a distinct file path.")
if set(input_paths.values()) & set(output_paths.values()):
    raise ValueError("An output must not overwrite one of its inputs.")
existing = [str(path) for path in output_paths.values() if path.exists()]
if existing:
    raise FileExistsError(f"Review existing outputs before replacing them: {existing}")

if set(EXPECTED_INPUT_SHA256) != set(INPUTS):
    raise ValueError("Every input must have its own expected SHA-256.")
input_fingerprints = {key: verify_input_sha256(path, EXPECTED_INPUT_SHA256[key])
                      for key, path in input_paths.items()}
inputs = {key: pd.read_csv(path, **INPUT_OPTIONS.get(key, {})) for key, path in input_paths.items()}
for key, frame in inputs.items():
    print({"input": key, "data_path": INPUTS[key], "rows": len(frame),
           "schema": {column: str(dtype) for column, dtype in frame.dtypes.items()},
           "sha256": input_fingerprints[key]})


{'input': 'g1_g6_current', 'data_path': 'data/processed/candidates_g1_g6_current.csv', 'rows': 1583, 'schema': {'candidate_key': 'object', 'candidate_name': 'object', 'county_city': 'object', 'office_type': 'object', 'electoral_district': 'object', 'G1_age_2026_minus_birth_year': 'object', 'birth_date': 'object', 'birth_year': 'object', 'G1_source_tier': 'object', 'G1_source': 'object', 'G2_sex': 'object', 'G2_source_tier': 'object', 'G2_source': 'object', 'G3_education_level': 'object', 'education_raw': 'object', 'G3_source': 'object', 'G3_note': 'object', 'education_source_tier': 'object', 'education_historical_level': 'object', 'G4_party': 'object', 'party_registration_raw': 'object', 'G4_source': 'object', 'G4_note': 'object', 'G5_first_time_any_elected_office': 'object', 'G5_label': 'object', 'G5_prior_election_year_example': 'object', 'G5_evidence': 'object', 'G5_source': 'object', 'G5_audit_status': 'object', 'G6_political_family': 'object', 'G6_label': 'object', 'G6_evidence': 

In [3]:
master = inputs["g1_g6_current"]
pending = inputs["identity_pending"]
assert master["candidate_key"].is_unique and len(master) == 1583
assert pending["candidate_key"].is_unique

CITIES = ["臺北市", "高雄市"]
council = master[master["county_city"].isin(CITIES) & master["office_type"].eq("直轄市議員")].copy()
print("council candidates:", council.groupby("county_city").size().to_dict())
assert council["G2_sex"].notna().all() and council["G4_party"].notna().all()

FIELDS = {
    "G1": ("G1_age_2026_minus_birth_year", "年齡（2026減出生年）", "high",
           "找出生年或出生日期；只有「現年XX歲」時填報導年齡並寫報導日期，age_derived_flag標示由報導年齡推算±1",
           "{name} {city}議員 參選人 歲 出生"),
    "G3": ("G3_education_level", "最高已完成學歷級別", "high",
           "填研究所／大學／大學以下；肄業、在學、就讀中、學分班、結業不算完成該學位",
           "{name} {city}議員 參選人 學歷 畢業"),
    "G5": ("G5_first_time_any_elected_office", "首次參選任何民選公職", "high",
           "1＝本人或媒體明確敘述首次參選；0＝找到曾參選任何民選公職（含里長、代表）的紀錄；「新人」不等於首次",
           "{name} 首次參選 OR 第一次參選 OR 里長 OR 代表"),
    "G6": ("G6_political_family", "配偶或三親等內親屬具公職", "low",
           "只在找到配偶或三親等內親屬（含姻親）任公職的證據時填1，並記親屬姓名、關係、職位；堂表兄弟姊妹為四親等不計；查無不可填0",
           "{name} 父親 OR 母親 OR 配偶 OR 代父出征 議員"),
}
for column, *_rest in FIELDS.values():
    assert column in council.columns

hint_birth = pending.set_index("candidate_key")["historical_birth_date"].to_dict()
rows = []
for _, r in council.iterrows():
    for field, (column, label, priority, rule, query) in FIELDS.items():
        if pd.notna(r[column]) and str(r[column]).strip() != "":
            continue
        hint = ""
        if r["candidate_key"] in hint_birth and field in ("G1", "G5"):
            hint = (f"同名同選區歷史參選紀錄，生日 {hint_birth[r['candidate_key']]}；"
                    "尚未確認同一人，確認後可支持G1與G5=0")
        rows.append({
            "candidate_key": r["candidate_key"], "candidate_name": r["candidate_name"],
            "county_city": r["county_city"], "electoral_district": r["electoral_district"],
            "G4_party": r["G4_party"], "field": field, "field_label": label,
            "priority": priority, "coding_rule": rule,
            "suggested_query": query.format(name=r["candidate_name"], city=r["county_city"]),
            "existing_hint": hint,
            "found_value": "", "age_derived_flag": "", "relative_detail": "",
            "source_url": "", "source_title_date": "", "source_quote": "",
            "source_type": "", "checked_by": "", "checked_date": "", "note": "",
        })
worklist = (pd.DataFrame(rows)
            .sort_values(["county_city", "electoral_district", "candidate_name", "field"], kind="stable")
            .reset_index(drop=True))

expected_rows = sum(int(council[c].isna().sum()) for c, *_ in FIELDS.values())
assert len(worklist) == expected_rows
assert not worklist.duplicated(["candidate_key", "field"]).any()
assert set(worklist["candidate_key"]) <= set(council["candidate_key"])
blank_cols = ["found_value", "age_derived_flag", "relative_detail", "source_url", "source_title_date",
              "source_quote", "source_type", "checked_by", "checked_date", "note"]
assert (worklist[blank_cols] == "").all().all()
print("rows:", len(worklist), "| candidates:", worklist["candidate_key"].nunique())
print(worklist.pivot_table(index="county_city", columns="field", values="candidate_key", aggfunc="count").to_string())
print("candidates needing any of G1/G3/G5:",
      worklist[worklist["priority"].eq("high")].groupby("county_city")["candidate_key"].nunique().to_dict())
print("rows with existing hint:", int((worklist["existing_hint"] != "").sum()))
outputs = {"manual_worklist": worklist}


council candidates: {'臺北市': 98, '高雄市': 101}
rows: 383 | candidates: 175
field        G1  G3  G5  G6
county_city                
臺北市          35  36  38  84
高雄市          34  33  32  91
candidates needing any of G1/G3/G5: {'臺北市': 44, '高雄市': 35}
rows with existing hint: 13


In [4]:
# Run only after the transformation cell has produced and checked `outputs`.
assert set(outputs) == set(OUTPUTS), "Produced datasets must match the declared output paths."
for key, frame in outputs.items():
    if not isinstance(frame, pd.DataFrame) or not frame.columns.is_unique:
        raise ValueError(f"Output {key} must be a DataFrame with unique column names.")

verification = {}
for key, frame in outputs.items():
    path = output_paths[key]
    path.parent.mkdir(parents=True, exist_ok=True)
    # Exclusive creation prevents an accidental overwrite during a rerun.
    with path.open("x", encoding="utf-8-sig", newline="") as stream:
        frame.to_csv(stream, index=False)
    reread = pd.read_csv(path, **OUTPUT_READ_OPTIONS.get(key, {}))
    assert list(reread.columns) == list(frame.columns), f"Schema changed while saving {key}."
    assert len(reread) == len(frame), f"Row count changed while saving {key}."
    # Adapt storage/type normalization explicitly when needed; preserve values.
    pd.testing.assert_frame_equal(frame.reset_index(drop=True), reread,
                                  check_dtype=False, check_categorical=False)
    verification[key] = {"data_path": OUTPUTS[key], "rows": len(reread),
                         "schema": {column: str(dtype) for column, dtype in reread.dtypes.items()},
                         "sha256": sha256_file(path), "status": "read-back checks passed"}
    print(verification[key])
assert all(sha256_file(path) == input_fingerprints[key] for key, path in input_paths.items()), "An input changed."
# Use these actual results to finish every declared processed-data document
# and Data Artifacts row. A failed write/read-back remains unverified.
# Finding notebooks must use the exact output path, matching role and SHA-256,
# never the first CSV, a raw source, or a fabricated example path.


{'data_path': 'data/processed/manual_check_worklist_taipei_kaohsiung_20261104.csv', 'rows': 383, 'schema': {'candidate_key': 'object', 'candidate_name': 'object', 'county_city': 'object', 'electoral_district': 'object', 'G4_party': 'object', 'field': 'object', 'field_label': 'object', 'priority': 'object', 'coding_rule': 'object', 'suggested_query': 'object', 'existing_hint': 'object', 'found_value': 'object', 'age_derived_flag': 'object', 'relative_detail': 'object', 'source_url': 'object', 'source_title_date': 'object', 'source_quote': 'object', 'source_type': 'object', 'checked_by': 'object', 'checked_date': 'object', 'note': 'object'}, 'sha256': 'fc88c9a7d26702c27d7ec1b1fd64d6e2d1129a1c1cc2fd0aa9339b45b00c69bc', 'status': 'read-back checks passed'}


## 工作單回填方式

- `found_value`：G1填出生年（或報導年齡）、G3填 研究所／大學／大學以下、G5填 1 或 0、G6填 1；查不到請留空，並在 `note` 寫查過哪些來源。
- `age_derived_flag`：只在用「現年XX歲」推算時填「由報導年齡推算±1」。
- `relative_detail`：G6專用，填「親屬姓名／關係／親等／公職」。
- `source_url`、`source_title_date`、`source_quote`：每個填入值都要有網址、標題日期與原文摘錄；`source_type` 填 官方／政黨／候選人自述／媒體。
- `checked_by`、`checked_date`：核對人與日期。
- 回填後不要直接改主表；整合時另以筆記本讀取回填工作單、驗證後產生主表新版本。
- 2026選舉公報預計投票日前約10天上線，屆時G1、G3可用官方公報覆核或補齊。
